In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

file_name = "EastWestAirlines.xlsx"

# Load the customer data sheet
df = pd.read_excel(file_name, sheet_name="data")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe().round(2))

In [ ]:
# Keep the original data unchanged
data = df.copy()

# Exclude the unique identifier
X = data.drop(columns=["ID#"])

# Convert all clustering features to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Fill missing numerical values if any exist
X = X.fillna(X.median())

print("Features used for clustering:")
print(X.columns.tolist())

print("\nShape after removing ID#:", X.shape)

In [ ]:
plt.figure(figsize=(15, 7))

sns.boxplot(data=X, orient="h")

plt.title("Boxplots of EastWest Airlines Customer Features")
plt.xlabel("Feature value")
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

print("Scaled data shape:", X_scaled.shape)

display(X_scaled.head())

In [ ]:
X.hist(figsize=(16, 12), bins=25, edgecolor="black")

plt.suptitle(
    "Distribution of EastWest Airlines Customer Features",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    X.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Airline Customer Features")
plt.tight_layout()
plt.show()

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.title("Elbow Method - EastWest Airlines")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.grid(True)

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(X_scaled)

data["KMeans_Cluster"] = kmeans_labels

print("Cluster sizes:")
print(data["KMeans_Cluster"].value_counts().sort_index())

kmeans_silhouette = silhouette_score(
    X_scaled,
    kmeans_labels
)

print(
    "K-Means Silhouette Score:",
    round(kmeans_silhouette, 4)
)

In [ ]:
# Step 8: Profile the K-Means clusters

# Add cluster labels to the original dataset
profile_df = df.copy()
profile_df["KMeans_Cluster"] = kmeans_labels

# Calculate average values for each cluster
cluster_profile = profile_df.drop(columns=["ID#"]).groupby(
    "KMeans_Cluster"
).mean().round(2)

print("Average customer characteristics by cluster:")
display(cluster_profile)

# Show customer count and percentage for each cluster
cluster_sizes = profile_df["KMeans_Cluster"].value_counts().sort_index()

cluster_summary = pd.DataFrame({
    "Customer_Count": cluster_sizes,
    "Percentage": (cluster_sizes / len(profile_df) * 100).round(2)
})

print("\nCluster size summary:")
display(cluster_summary)

In [ ]:
# Step : Hierarchical Clustering

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd

# Test different linkage methods
linkage_methods = ["ward", "complete", "average"]
hierarchical_results = []

for linkage in linkage_methods:
    model = AgglomerativeClustering(
        n_clusters=4,
        linkage=linkage
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)

    hierarchical_results.append({
        "Linkage": linkage,
        "Silhouette_Score": round(score, 4)
    })

    print(f"\nLinkage: {linkage}")
    print("Cluster sizes:")
    print(pd.Series(labels).value_counts().sort_index())
    print("Silhouette Score:", round(score, 4))

# Store results in a DataFrame
hierarchical_comparison = pd.DataFrame(hierarchical_results)

print("\nHierarchical Clustering Comparison:")
display(hierarchical_comparison)

In [ ]:
# Compare K-Means with the hierarchical methods

comparison_data = [
    {
        "Algorithm": "K-Means",
        "Method": "K-Means",
        "Silhouette_Score": 0.3092
    }
]

for _, row in hierarchical_comparison.iterrows():
    comparison_data.append({
        "Algorithm": "Hierarchical",
        "Method": row["Linkage"],
        "Silhouette_Score": row["Silhouette_Score"]
    })

comparison_df = pd.DataFrame(comparison_data)

print("Clustering Algorithm Comparison:")
display(
    comparison_df.sort_values(
        by="Silhouette_Score",
        ascending=False
    ).reset_index(drop=True)
)

In [ ]:
# : Find a suitable DBSCAN eps value

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

# Find the distance to the 5th nearest neighbor
neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(X_scaled)

distances, indices = neighbors.kneighbors(X_scaled)

# Sort distances to visualize the k-distance curve
k_distances = np.sort(distances[:, 4])

plt.figure(figsize=(10, 5))
plt.plot(k_distances)
plt.title("DBSCAN 5-Nearest-Neighbor Distance Plot")
plt.xlabel("Customers (sorted by distance)")
plt.ylabel("Distance to 5th nearest neighbor")
plt.grid(True)
plt.show()

In [ ]:
# Step : Experiment with DBSCAN parameters

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

eps_values = [1.0, 1.5, 2.0, 2.5, 3.0]
min_samples_values = [5, 10]

dbscan_results = []

for eps in eps_values:
    for min_samples in min_samples_values:

        model = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        labels = model.fit_predict(X_scaled)

        # -1 represents noise
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_count = np.sum(labels == -1)

        # Silhouette score only if at least 2 non-noise clusters exist
        non_noise_mask = labels != -1
        non_noise_labels = labels[non_noise_mask]
        non_noise_data = X_scaled[non_noise_mask]

        if (
            len(set(non_noise_labels)) >= 2
            and len(non_noise_labels) > len(set(non_noise_labels))
        ):
            score = silhouette_score(
                non_noise_data,
                non_noise_labels
            )
        else:
            score = np.nan

        dbscan_results.append({
            "eps": eps,
            "min_samples": min_samples,
            "Clusters": n_clusters,
            "Noise_Points": int(noise_count),
            "Silhouette_Score": (
                round(score, 4) if not np.isnan(score) else None
            )
        })

dbscan_comparison = pd.DataFrame(dbscan_results)

print("DBSCAN Parameter Comparison:")
display(dbscan_comparison)

In [ ]:
# Step : Visualize all three clustering algorithms using PCA

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, DBSCAN

# Reduce the scaled data to two dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(
    "Variance explained by the first two components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

# Recreate Average-linkage hierarchical labels
hierarchical_model = AgglomerativeClustering(
    n_clusters=4,
    linkage="average"
)
hierarchical_labels = hierarchical_model.fit_predict(X_scaled)

# Recreate DBSCAN labels using the selected parameters
dbscan_model = DBSCAN(
    eps=2.5,
    min_samples=5
)
dbscan_labels = dbscan_model.fit_predict(X_scaled)

# Visualize each algorithm
models = [
    ("K-Means Clustering", kmeans_labels),
    ("Hierarchical Clustering (Average)", hierarchical_labels),
    ("DBSCAN (eps=2.5, min_samples=5)", dbscan_labels)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, labels) in zip(axes, models):
    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        cmap="tab10",
        s=12,
        alpha=0.7
    )

    ax.set_title(title)
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")

plt.tight_layout()
plt.show()

In [ ]:
# Step : Final comparison of clustering algorithms

from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

def evaluate_clustering(name, labels, X, exclude_noise=False):
    labels = np.asarray(labels)

    if exclude_noise:
        mask = labels != -1
        labels_to_score = labels[mask]
        X_to_score = X[mask]
    else:
        labels_to_score = labels
        X_to_score = X

    n_clusters = len(set(labels_to_score))
    score = np.nan

    if 2 <= n_clusters < len(labels_to_score):
        score = silhouette_score(X_to_score, labels_to_score)

    return {
        "Algorithm": name,
        "Clusters": len(set(labels) - {-1}),
        "Noise_Points": int(np.sum(labels == -1)),
        "Silhouette_Score": (
            round(score, 4) if not np.isnan(score) else None
        )
    }

final_results = [
    evaluate_clustering(
        "K-Means (K=4)",
        kmeans_labels,
        X_scaled
    ),
    evaluate_clustering(
        "Hierarchical (Ward)",
        AgglomerativeClustering(
            n_clusters=4,
            linkage="ward"
        ).fit_predict(X_scaled),
        X_scaled
    ),
    evaluate_clustering(
        "Hierarchical (Complete)",
        AgglomerativeClustering(
            n_clusters=4,
            linkage="complete"
        ).fit_predict(X_scaled),
        X_scaled
    ),
    evaluate_clustering(
        "Hierarchical (Average)",
        hierarchical_labels,
        X_scaled
    ),
    evaluate_clustering(
        "DBSCAN (eps=2.5, min_samples=5)",
        dbscan_labels,
        X_scaled,
        exclude_noise=True
    )
]

final_comparison = pd.DataFrame(final_results)

print("Final Clustering Comparison:")
display(
    final_comparison.sort_values(
        by="Silhouette_Score",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)
)

In [ ]:
# Step : Export results

results_df = df.copy()
results_df["KMeans_Cluster"] = kmeans_labels
results_df["Hierarchical_Average_Cluster"] = hierarchical_labels
results_df["DBSCAN_Cluster"] = dbscan_labels

results_df.to_csv("EastWestAirlines_Cluster_Results.csv", index=False)
final_comparison.to_csv("Clustering_Model_Comparison.csv", index=False)

print("Files saved successfully.")

In [ ]:
from google.colab import files

files.download("EastWestAirlines_Cluster_Results.csv")
files.download("Clustering_Model_Comparison.csv")

In [ ]:

import pandas as pd
import numpy as np

# Create a copy of the original data
profile_df = df.copy()

# Add cluster labels from each algorithm
profile_df["KMeans_Cluster"] = kmeans_labels
profile_df["Hierarchical_Cluster"] = hierarchical_labels
profile_df["DBSCAN_Cluster"] = dbscan_labels

# Exclude the ID column because it is not a customer characteristic
features = [col for col in df.columns if col != "ID#"]

# K-Means cluster sizes
print("K-MEANS CLUSTER SIZES")
print(profile_df["KMeans_Cluster"].value_counts().sort_index())

# K-Means cluster profiles
kmeans_profile = profile_df.groupby("KMeans_Cluster")[features].mean().round(2)
print("\nK-MEANS CLUSTER PROFILE (MEAN VALUES)")
display(kmeans_profile)

# Hierarchical cluster sizes and profiles
print("\nHIERARCHICAL CLUSTER SIZES")
print(profile_df["Hierarchical_Cluster"].value_counts().sort_index())

hierarchical_profile = profile_df.groupby("Hierarchical_Cluster")[features].mean().round(2)
print("\nHIERARCHICAL CLUSTER PROFILE")
display(hierarchical_profile)

# DBSCAN cluster sizes and profiles
print("\nDBSCAN CLUSTER SIZES (including -1 noise)")
print(profile_df["DBSCAN_Cluster"].value_counts().sort_index())

dbscan_profile = profile_df.groupby("DBSCAN_Cluster")[features].mean().round(2)
print("\nDBSCAN CLUSTER PROFILE")
display(dbscan_profile)


In [ ]:

from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

# Calculate silhouette scores using the scaled features
kmeans_score = silhouette_score(X_scaled, kmeans_labels)
hierarchical_score = silhouette_score(X_scaled, hierarchical_labels)

# DBSCAN silhouette score: exclude noise points (-1)
dbscan_mask = np.asarray(dbscan_labels) != -1
dbscan_valid_labels = np.asarray(dbscan_labels)[dbscan_mask]

if len(set(dbscan_valid_labels)) >= 2 and len(set(dbscan_valid_labels)) < len(dbscan_valid_labels):
    dbscan_score = silhouette_score(
        np.asarray(X_scaled)[dbscan_mask],
        dbscan_valid_labels
    )
else:
    dbscan_score = np.nan

comparison_df = pd.DataFrame({
    "Algorithm": [
        "K-Means (k=4)",
        "Hierarchical (Average)",
        "DBSCAN (eps=2.5, min_samples=5)"
    ],
    "Silhouette Score": [
        kmeans_score,
        hierarchical_score,
        dbscan_score
    ],
    "Number of Clusters": [
        len(set(kmeans_labels)),
        len(set(hierarchical_labels)),
        len(set(dbscan_valid_labels))
    ],
    "Noise Points": [
        0,
        0,
        int(np.sum(~dbscan_mask))
    ]
})

display(comparison_df.round(4))


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.bar(
    comparison_df["Algorithm"],
    comparison_df["Silhouette Score"]
)

plt.title("Comparison of Clustering Algorithms")
plt.xlabel("Clustering Algorithm")
plt.ylabel("Silhouette Score")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


### Model Comparison and Evaluation

The clustering algorithms were evaluated using the silhouette score, which measures how well-separated and cohesive the clusters are. A higher score generally indicates better separation.

K-Means was evaluated using four clusters. Hierarchical clustering was evaluated using average linkage with four clusters. DBSCAN was evaluated using `eps = 2.5` and `min_samples = 5`.

The silhouette score of DBSCAN was calculated after excluding noise points. Therefore, its score should be interpreted carefully when comparing it with the other algorithms.

The final model selection should consider the silhouette score, the distribution of customers across clusters, and whether the resulting groups are meaningful for customer segmentation. A high silhouette score alone does not guarantee the most useful model.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Reduce scaled data to two dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Create plots for all three algorithms
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cluster_results = [
    ("K-Means Clustering", kmeans_labels),
    ("Hierarchical Clustering", hierarchical_labels),
    ("DBSCAN Clustering", dbscan_labels)
]

for ax, (title, labels) in zip(axes, cluster_results):
    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        cmap="viridis",
        s=10,
        alpha=0.7
    )
    ax.set_title(title)
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")

plt.suptitle("Customer Clustering Visualization Using PCA")
plt.tight_layout()
plt.show()

print("Explained variance by PC1 and PC2:",
      np.round(pca.explained_variance_ratio_, 4))
print("Total variance explained:",
      round(pca.explained_variance_ratio_.sum(), 4))


## PCA Visualization and Interpretation

Principal Component Analysis (PCA) was used to reduce the scaled customer data to two dimensions for visualization.

The scatter plots display the clustering results obtained from K-Means, Hierarchical Clustering, and DBSCAN. Each color represents a cluster assigned by the respective algorithm.

The plots help visually compare the grouping patterns produced by the algorithms. DBSCAN may also identify noise points that do not belong to any dense cluster.

PCA provides a simplified view of the original multidimensional data. Therefore, the visual separation observed in these plots may not fully represent the clustering structure in the original feature space.

## Statistical Findings

The East-West Airlines dataset was analyzed using three clustering techniques: K-Means, Hierarchical Clustering, and DBSCAN.

1. **K-Means Clustering:** The model with four clusters divided the customers into groups of different sizes. Cluster profiling can help identify differences in customer balances, bonus miles, flight activity, and membership duration.

2. **Hierarchical Clustering:** Different linkage methods were evaluated. Average linkage achieved a high silhouette score, but its cluster distribution was highly imbalanced, with most customers assigned to one cluster.

3. **DBSCAN Clustering:** Different values of epsilon and minimum samples were tested. The model identified dense groups of customers and also classified some customers as noise.

4. **Model Evaluation:** The silhouette score was used to assess cluster separation. However, cluster sizes, customer characteristics, and practical usefulness must also be considered when comparing the models.

Overall, the analysis demonstrates how clustering algorithms can be used to explore customer segments in airline loyalty data.

## Conclusion

This case study applied K-Means, Hierarchical Clustering, and DBSCAN to the East-West Airlines customer dataset to explore customer segmentation.

The analysis included data inspection, preprocessing, feature scaling, exploratory data analysis, clustering, silhouette score evaluation, and PCA visualization.

The results demonstrate that different clustering algorithms can produce different customer groupings. Although silhouette scores are useful for evaluation, a high score alone does not guarantee that the resulting clusters are meaningful or suitable for business use.

The identified customer groups can help the airline explore differences in travel activity, loyalty-program participation, bonus miles, and account balances. These insights may support more targeted marketing campaigns and customer engagement strategies.

Further analysis and business validation would be required before using the clusters for operational decisions.